This notebook may not be fully functional and may be very messy<br>
this was used as a 'testing' file and did not have the full intention of it being used for much<br><br>

if any cells cause an error, it may be that specific test was not fully fixed, proceeding further may produce working cells again since concepts may be disjoint<br>
This was also being adjusted for the transition from MedMentions data to PubMed data, as such has not been checked since that it still fully works<br>

it does include code for training reranker and 'verifier' model<br>
the verifier model is not used

Pre-Requisites

In [1]:
!pip install bioc faiss-gpu-cu12

  Using cached bioc-2.1-py3-none-any.whl.metadata (4.6 kB)
  Using cached faiss_gpu_cu12-1.14.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
  Using cached jsonlines-4.0.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached intervaltree-3.2.1-py2.py3-none-any.whl.metadata (12 kB)
  Using cached docopt-0.6.2-py2.py3-none-any.whl
  Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (62 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
Using cached bioc-2.1-py3-none-any.whl (33 kB)
Using cached faiss_gpu_cu12-1.14.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (48.4 MB)
Using cached jsonlines-4.0.0-py3-none-any.whl (8.7 kB)
Using cached numpy-2.2.6-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (16.8 MB)
Using cached intervaltree-3.2.1-py2.py3-none-any.whl (25 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Attempting uninstall: 

Download files from google drive

In [2]:
import faiss

print(faiss.__version__)
print(faiss.__file__)
print("GPU support:", hasattr(faiss, "StandardGpuResources"))

1.14.1
/opt/miniconda3/lib/python3.10/site-packages/faiss/__init__.py
GPU support: True


In [3]:
!pip uninstall -y faiss-cpu faiss-gpu-cu12
!pip install faiss-gpu-cu12

Found existing installation: faiss-gpu-cu12 1.14.1.post1
Uninstalling faiss-gpu-cu12-1.14.1.post1:
  Successfully uninstalled faiss-gpu-cu12-1.14.1.post1
  Using cached faiss_gpu_cu12-1.14.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (13 kB)
Using cached faiss_gpu_cu12-1.14.1.post1-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (48.4 MB)


In [4]:
from pathlib import Path
# from google.colab import drive

# drive.mount('/content/drive')

# DATA_DIR = Path('/content/drive/My Drive/Colab Notebooks/Internship')
DATA_DIR = Path('')

In [5]:
from bioc import biocxml
import gzip

def load_bioc(path):
    with gzip.open(path, "rt", encoding="utf-8") as fp:
        collection = biocxml.load(fp)
    return collection.documents

In [6]:
def extract_text(paper):
    passages = paper[0]["documents"][0]["passages"]
    text = ""
    for passage in passages:
        text += passage["text"] + "\n"

    return text

In [7]:
# print(extract_text(paper))

In [8]:
# papers[2]

In [9]:
from bioc import biocxml
import gzip

def load_bioc(path):
    with gzip.open(path, "rt", encoding="utf-8") as fp:
        collection = biocxml.load(fp)
    return collection.documents

In [10]:
def load_medmentions():
  DATA_DIR = 'Data/'
  TRAIN_FILE = DATA_DIR + "MedMentions/medmentions_st21pv_train.bioc.xml.gz"
  VAL_FILE = DATA_DIR + "MedMentions/medmentions_st21pv_val.bioc.xml.gz"
  TEST_FILE = DATA_DIR + "MedMentions/medmentions_st21pv_test.bioc.xml.gz"

  train_docs = load_bioc(TRAIN_FILE)
  val_docs = load_bioc(VAL_FILE)
  test_docs = load_bioc(TEST_FILE)

  print(f"{len(train_docs)} training documents")
  print(f"{len(val_docs)} validation documents")
  print(f"{len(test_docs)} test documents")

  return train_docs, val_docs, test_docs

In [11]:
train_docs, val_docs, test_docs = load_medmentions()

2635 training documents
878 validation documents
879 test documents


Get UMLS

In [12]:
def get_UMLS_files():
  UMLS_FOLDER = "Data/Ontology/"

  MRCONSO_FILE = UMLS_FOLDER + "MRCONSO.RRF"
  MRDEF_FILE = UMLS_FOLDER + "MRDEF.RRF"
  MRSTY_FILE = UMLS_FOLDER + "MRSTY.RRF"

  return MRCONSO_FILE, MRDEF_FILE, MRSTY_FILE

In [13]:
# UMLS has vastly more CUIS than medmentions uses, so to save time, we only need to consider the ones it uses instead of them all
def get_medmentions_cuis():
  used_cuis = set()

  for doc in train_docs + val_docs + test_docs:
      for passage in doc.passages:
          for anno in passage.annotations:
              cui = anno.infons.get("concept_id")
              if cui:
                  used_cuis.add(cui)

  return used_cuis

In [14]:
used_cuis = get_medmentions_cuis()

In [16]:
from collections import defaultdict
from tqdm import tqdm

def prep_UMLS_concepts(MRCONSO_FILE):
  concepts = defaultdict(lambda: {
    "name": None,
    "aliases": set(),
    "name_score": None
  })

  with open(MRCONSO_FILE, encoding="utf-8") as f:
      for line in tqdm(f, desc="Loading MRCONSO"):
          fields = line.rstrip("\n").split("|")

          cui = fields[0]
          lang = fields[1]
          term_type = fields[12]
          term = fields[14]

          if lang != "ENG":
              continue
          if not term.strip():
              continue

          concepts[cui]["aliases"].add(term)

          if (
              concepts[cui]["name"] is None
              and term_type == "PN"
          ):
              concepts[cui]["name"] = term
              
  return concepts

In [17]:
# currently unused, it is the definitions of each cui

def prep_UMLS_defs(MRDEF_FILE):
  definitions = {}

  with open(MRDEF_FILE, encoding="utf-8") as f:
      for line in tqdm(f, desc="Loading MRDEF"):
          fields = line.rstrip("\n").split("|")

          cui = fields[0]
          definition = fields[5]

          if cui in used_cuis:
            continue

          # keep the first definition
          if cui not in definitions:
              definitions[cui] = definition
  return definitions

In [18]:
def prep_UMLS_sty(MRSTY_FILE):
    semantic_types = {}

    with open(MRSTY_FILE, encoding="utf-8") as f:
        for line in tqdm(f, desc="Loading MRSTY"):
            fields = line.rstrip("\n").split("|")

            cui = fields[0]
            sty = fields[3]

            if cui in used_cuis:
                continue
            if cui in semantic_types:
                semantic_types[cui].append(sty)
            else:
                semantic_types[cui] = [sty]
    return semantic_types

In [19]:
def generate_ontology():
  MRCONSO_FILE, MRDEF_FILE, MRSTY_FILE = get_UMLS_files()
  concepts = prep_UMLS_concepts(MRCONSO_FILE)
  # definitions = prep_UMLS_defs(MRDEF_FILE)
  semantic_types = prep_UMLS_sty(MRSTY_FILE)

  ontology = []

  for cui, concept in concepts.items():
      ontology.append({
          "id": f"UMLS:{cui}",
          "name": concept["name"],
          "aliases": sorted(concept["aliases"]),
          # "definition": definitions.get(cui, ""),
          "types": semantic_types.get(cui, [])
      })
  return ontology

In [20]:
ontology = generate_ontology()

Loading MRCONSO: 18064970it [00:16, 1092435.50it/s]
Loading MRSTY: 3876927it [00:04, 860931.53it/s] 


In [21]:
import json

from sentence_transformers import SentenceTransformer
from pathlib import Path
import numpy as np
import faiss
import torch

SAPBERT_MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"
CANDIDATES_TO_GENERATE = 10

CACHE_DIRECTORY = Path("Cache/SapBERT")
EMBEDDINGS_PATH = CACHE_DIRECTORY / "umls_embeddings.npy"
ONTOLOGY_PATH = CACHE_DIRECTORY / "umls_ontology.json"


def get_sapbert_model(model_name=SAPBERT_MODEL_NAME):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"loading sapbert on {device}")

    return SentenceTransformer(model_name, device=device)


def encode_entities(entity_names, model, batch_size=32):
    embeddings = model.encode(
        entity_names,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.array(embeddings)


def rank_entities(entity_vectors, index, candidates_to_generate):
    return index.search(entity_vectors, k=candidates_to_generate)


def generate_sapbert_scores_indices(entity_texts,
                                    index,
                                    model,
                                    candidates_to_generate=CANDIDATES_TO_GENERATE
                                    ):
   cleaned_entity_texts = [
        cleanse_anno(text)
        for text in entity_texts
    ]
   
   entity_vectors = encode_entities(cleaned_entity_texts, model)

   return rank_entities(entity_vectors, index, candidates_to_generate) # scores, indices


def make_training_scores(ontology, model):
    train_docs, val_docs, test_docs = load_medmentions()

    entity_texts = make_entity_texts(
        train_docs + val_docs + test_docs
    )

    index = build_sapbert_index(ontology, model)

    return  generate_sapbert_scores_indices(
        entity_texts,
        index,
    )


def make_scores(ner_predictions, model):
    entity_texts = [
        cleanse_anno(prediction["text"])
        for prediction in ner_predictions
    ]

    index = build_sapbert_index()

    return generate_sapbert_scores_indices(
       entity_texts, 
       index,
       model
       )


def load_sapbert_index(
    embeddings_path=EMBEDDINGS_PATH,
    ontology_path=ONTOLOGY_PATH
):
    ontology_vectors = np.load(embeddings_path)

    with open(ontology_path, "r", encoding="utf-8") as file:
        ontology = json.load(file)

    if len(ontology_vectors) != len(ontology):
        raise ValueError(
            "Ontology and embedding cache lengths do not match: "
            f"{len(ontology_vectors)} vectors vs "
            f"{len(ontology)} entities"
        )

    index = create_faiss_index(ontology_vectors)

    print(
        f"Loaded {len(ontology_vectors):,} cached ontology "
        "embeddings into FAISS"
    )

    return ontology, index


def build_sapbert_index(ontology, model):
    entity_names = [entity["name"] for entity in ontology]

    ontology_vectors = encode_entities(entity_names)

    return create_faiss_index(ontology_vectors)


def get_or_build_sapbert_index(ontology, model):
    cache_exists = (
        EMBEDDINGS_PATH.exists()
        and ONTOLOGY_PATH.exists()
    )

    if cache_exists:
        return load_sapbert_index()

    print("No cached SapBERT ontology index found.")
    print("Encoding ontology for the first time...")

    return build_and_save_sapbert_index(ontology, model)

def build_and_save_sapbert_index(
    ontology,
    model,
    embeddings_path=EMBEDDINGS_PATH,
    ontology_path=ONTOLOGY_PATH
):
    CACHE_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True
    )

    entity_names = [entity["name"] for entity in ontology]

    ontology_vectors = encode_entities(
        entity_names,
        model,
        batch_size=32
    )

    ontology_vectors = np.asarray(
        ontology_vectors,
        dtype=np.float32
    )

    np.save(
        embeddings_path,
        ontology_vectors
    )

    with open(ontology_path, "w", encoding="utf-8") as file:
        json.dump(
            ontology,
            file,
            ensure_ascii=False
        )

    index = create_faiss_index(ontology_vectors)

    print(f"Saved {len(ontology_vectors):,} ontology embeddings to {embeddings_path}")

    return ontology, index

def create_faiss_index(ontology_vectors):
    ontology_vectors = np.asarray(
        ontology_vectors,
        dtype=np.float32
    )

    ontology_vectors = np.ascontiguousarray(ontology_vectors)

    dimension = ontology_vectors.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(ontology_vectors)

    return index

In [22]:
def move_index_to_gpu(index, gpu_id=0):
    resources = faiss.StandardGpuResources()

    gpu_index = faiss.index_cpu_to_gpu(resources, gpu_id, index)

    return gpu_index

In [23]:
sapbert_model = get_sapbert_model()

ontology, sapbert_index = get_or_build_sapbert_index(
    ontology,
    sapbert_model
)

No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


loading sapbert on cuda
Loaded 3,529,965 cached ontology embeddings into FAISS


In [24]:
sapbert_index = move_index_to_gpu(sapbert_index)

In [25]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: NVIDIA RTX A6000


In [26]:
print(next(sapbert_model.parameters()).device)

cuda:0


In [27]:
# concepts["C0010674"]

In [29]:
# for i in range(len(ontology)):
#     if ontology[i]["types"]:
#         print(ontology[i])
#         break

In [30]:
# ontology[7]

In [31]:
# # also currently unused

# ontology_ids = {entry["id"] for entry in ontology}

# for doc in train_docs + val_docs + test_docs:
#     for passage in doc.passages:
#         passage.annotations = [
#             anno
#             for anno in passage.annotations
#             if anno.infons.get("concept_id") in ontology_ids
#         ]

First stage candidate retrieval

In [32]:
from sentence_transformers import SentenceTransformer

MODEL_NAME = "cambridgeltl/SapBERT-from-PubMedBERT-fulltext"

model = SentenceTransformer(MODEL_NAME)

No sentence-transformers model found with name cambridgeltl/SapBERT-from-PubMedBERT-fulltext. Creating a new one with mean pooling.


Entities in ontology

In [33]:
entities = [ e['name'] for e in ontology ]
len(entities)

3529965

Medmentions Entities

In [28]:
def cleanse_anno(text):
    text = (
      (text).replace("α", "alpha")
      .replace("β", "beta")
      .replace("γ", "gamma")
      .replace("δ", "delta")
      .replace("κ", "kappa")
      .replace("λ", "lambda")
      .replace("μ", "mu")
      .replace("ω", "omega")
      .replace(" ii ", " 2 ")
      .replace("(ii)", "(2)")
      .replace("ii ", "2 ", 1)
      .replace(" iii ", " 3 ")
      .replace("(iii)", "(3)")
      .replace("iii ", "3 ", 1)
              )
  
    if text.endswith(" ii"):
      text = " 2".join(text.rsplit(" ii", 1))
    if text.endswith(" iii"):
      text = " 3".join(text.rsplit(" iii", 1))

    return text

In [1]:
def make_entity_texts(train_docs, val_docs, test_docs):
    entity_texts = []
    
    for doc in train_docs+val_docs+test_docs:
      for passage in doc.passages:
        for anno in passage.annotations:
          anno.text = cleanse_anno(anno.text)
          entity_texts.append(anno.text)
    
    return sorted(set(entity_texts)) 

In [2]:
entity_texts = make_entity_texts(train_docs, val_docs, test_docs)

NameError: name 'train_docs' is not defined

In [37]:
sapbert_scores, sapbert_indices = generate_sapbert_scores_indices(entity_texts, sapbert_index, model)

Batches:   0%|          | 0/2061 [00:00<?, ?it/s]

Nearest matches

In [4]:
from collections import Counter, defaultdict
import re
import json
import unicodedata
import numpy as np
from pathlib import Path
from pyterrier_pisa import PisaIndex

CACHE_DIRECTORY = Path("Cache/ngram")
INDEX_PATH = CACHE_DIRECTORY / "umls_6gram_pisa"
MAPPING_PATH = (CACHE_DIRECTORY / "alias_docno_to_concept_idx.npy")
METADATA_PATH = CACHE_DIRECTORY / "metadata.json"

ALIASES_TO_RETRIEVE = 50
CANDIDATES_TO_GENERATE = 10
NGRAM_SIZE = 6

def normalize_for_ngrams(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


def make_char_ngrams(text, n=NGRAM_SIZE):
    text = normalize_for_ngrams(text)

    if not text:
        return []

    padding = " " * (n - 1)
    padded = f"{padding}{text}{padding}"

    return [
        padded[i:i + n].replace(" ", "_")
        for i in range(len(padded) - (n-1))
    ]


def make_ngram_weights(text):
    return dict(Counter(make_char_ngrams(text)))

# go through ontology, assign every entity a doc number, and a dict of its n-gram weights
def get_alias_docs(ontology):
    alias_documents = []
    alias_docno_to_concept_idx = []
    
    for concept_idx, concept in enumerate(ontology):
        terms = [concept["name"], *concept.get("aliases", [])]
    
        seen_terms = set()
    
        for term in terms:
            if not term:
                continue
    
            normalized = normalize_for_ngrams(term)
    
            if normalized in seen_terms:
                continue
    
            seen_terms.add(normalized)
    
            ngram_weights = make_ngram_weights(term)
    
            if not ngram_weights:
                continue
    
            docno = str(len(alias_documents))
    
            alias_documents.append({
                "docno": docno,
                "toks": ngram_weights,
            })
    
            alias_docno_to_concept_idx.append(concept_idx)
            
    return alias_documents, alias_docno_to_concept_idx


def build_ngram_index(ontology, index_path=INDEX_PATH,):
    CACHE_DIRECTORY.mkdir(
        parents=True,
        exist_ok=True,
    )

    alias_documents, alias_docno_to_concept_idx = get_alias_docs(ontology)

    pisa_index = PisaIndex(str(index_path), stemmer="none", overwrite=True)

    indexer = pisa_index.toks_indexer(text_field="toks", mode="overwrite")

    indexer.index(alias_documents)

    alias_docno_to_concept_idx = np.asarray(alias_docno_to_concept_idx, dtype=np.int64)

    np.save(MAPPING_PATH, alias_docno_to_concept_idx)

    save_ngram_cache_metadata(ontology)

    print(f"Saved {len(alias_docno_to_concept_idx):,} aliases to the cached {NGRAM_SIZE}-gram index")

    return pisa_index, alias_docno_to_concept_idx

def make_ngram_queries(entity_texts):
    queries = [
        {
            "qid": str(i),
            "query_toks": make_ngram_weights(entity_text),
        }
        for i, entity_text in enumerate(entity_texts)
    ]
    return queries

def get_ngram_results(pisa_index, queries, aliases_to_retrieve=50):
    
    retriever = pisa_index.quantized(
        num_results=aliases_to_retrieve,
        threads=8,
    )
    
    return retriever(queries)

def pisa_results_to_candidate_arrays(
    results,
    alias_docno_to_concept_idx, 
    num_queries,
    top_k=10,
):
    # make two arrays with default values
    scores = np.full(
        (num_queries, top_k),
        -np.inf,
        dtype=np.float32,
    )

    indices = np.full(
        (num_queries, top_k),
        -1,
        dtype=np.int64,
    )

    results_by_qid = defaultdict(list)

    for result in results:
        results_by_qid[str(result["qid"])].append(result)

    for qid, group in results_by_qid.items():
        mention_idx = int(qid)

        group = sorted(
            group,
            key=lambda result: result["rank"],
        )

        seen_concepts = set()
        candidate_position = 0

        for result in group:
            alias_idx = int(result["docno"])
            concept_idx = alias_docno_to_concept_idx[alias_idx]

            if concept_idx in seen_concepts:
                continue

            seen_concepts.add(concept_idx)

            scores[mention_idx, candidate_position] = float(
                result["score"]
            )

            indices[mention_idx, candidate_position] = concept_idx

            candidate_position += 1

            if candidate_position >= top_k:
                break

    return scores, indices

def generate_ngram_scores_and_indices(entity_texts,
    pisa_index,
    alias_docno_to_concept_idx,
    candidates_to_generate=CANDIDATES_TO_GENERATE):

    queries = make_ngram_queries(entity_texts)

    results = get_ngram_results(pisa_index, queries)

    ngram_scores, ngram_indices = pisa_results_to_candidate_arrays(
        results,
        alias_docno_to_concept_idx, 
        num_queries=len(entity_texts),
        top_k=candidates_to_generate,
    )
    
    return ngram_scores, ngram_indices

def get_training_ngram_scores_and_indices(
        pisa_index,
        alias_docno_to_concept_idx,):

    train_docs, val_docs, test_docs = load_medmentions()

    entity_texts = make_entity_texts(train_docs+val_docs+test_docs)

    ngram_scores, ngram_indices = generate_ngram_scores_and_indices(
        entity_texts,
        pisa_index,
        alias_docno_to_concept_idx,
    )
    
    return ngram_scores, ngram_indices

def get_ngram_cache_metadata(ontology):
    alias_count = sum(
        1 + len(concept.get("aliases", []))
        for concept in ontology
    )

    return {
        "ngram_size": NGRAM_SIZE,
        "ontology_concept_count": len(ontology),
        "raw_alias_count": alias_count,
    }


def save_ngram_cache_metadata(ontology):
    metadata = get_ngram_cache_metadata(ontology)

    with open(METADATA_PATH, "w", encoding="utf-8") as file:
        json.dump(
            metadata,
            file,
            indent=2,
        )


def ngram_cache_is_compatible(ontology):
    if not METADATA_PATH.exists():
        return False

    try:
        with open(METADATA_PATH, "r", encoding="utf-8") as file:
            cached_metadata = json.load(file)
    except (OSError, json.JSONDecodeError):
        return False

    expected_metadata = get_ngram_cache_metadata(ontology)

    return cached_metadata == expected_metadata

def load_ngram_index(
    index_path=INDEX_PATH,
    mapping_path=MAPPING_PATH,
):
    if not index_path.exists():
        raise FileNotFoundError(f"N-gram PISA index was not found at {index_path}")

    if not mapping_path.exists():
        raise FileNotFoundError(f"N-gram alias mapping was not found at {mapping_path}")

    pisa_index = PisaIndex(
        str(index_path),
        stemmer="none",
    )

    alias_docno_to_concept_idx = np.load(mapping_path)

    alias_docno_to_concept_idx = np.asarray(
        alias_docno_to_concept_idx,
        dtype=np.int64,
    )

    print(f"Loaded cached {NGRAM_SIZE}-gram index with {len(alias_docno_to_concept_idx):,} aliases")

    return pisa_index, alias_docno_to_concept_idx

def get_or_build_ngram_index(ontology):
    cache_exists = (
        INDEX_PATH.exists()
        and MAPPING_PATH.exists()
        and METADATA_PATH.exists()
    )

    if cache_exists and ngram_cache_is_compatible(ontology):
        return load_ngram_index()

    if cache_exists:
        print("Cached n-gram index is incompatible with the current ontology or n-gram settings.")
    else:
        print("No cached n-gram index found.")

    print("Building n-gram index for the first time...")

    return build_ngram_index(ontology)

In [46]:
pisa_index, alias_docno_to_concept_idx = (get_or_build_ngram_index(ontology))
ngram_score, ngram_indices = generate_ngram_scores_and_indices(entity_texts, pisa_index, alias_docno_to_concept_idx)

Loaded cached 6-gram index with 8,515,904 aliases


Combine n-gram results with the sapbert results using RRF

In [47]:
def combine_candidates(ngram_scores, ngram_indices, sap_scores, sap_indices):
    combined_scores = np.concatenate((ngram_scores, sap_scores), axis=1)
    combined_indices =  np.concatenate((ngram_indices, sap_indices), axis=1)

    return combined_scores, combined_indices

In [50]:
combined_scores, combined_indices = combine_candidates(ngram_score, ngram_indices, sapbert_scores, sapbert_indices)

In [51]:
# candidate_lookup = { entity_text:combined_indices[i].tolist() for i,entity_text in enumerate(entity_texts) }

In [52]:
# combined_scores.shape

In [53]:
def rrf_top_k(
    ngram_indices,
    sapbert_indices,
    top_k=5,
    rrf_k=10.0,
    ngram_weight=0.5,
    sapbert_weight=1.0,
):
    ngram_indices = np.asarray(ngram_indices)
    sapbert_indices = np.asarray(sapbert_indices)

    if ngram_indices.shape != sapbert_indices.shape:
        raise ValueError("ngram_indices and sapbert_indices must have the same shape.")

    n_rows, candidates_per_method = ngram_indices.shape

    output_indices = np.empty(
        (n_rows, top_k),
        dtype=ngram_indices.dtype,
    )
    output_scores = np.empty(
        (n_rows, top_k),
        dtype=np.float32,
    )

    # rrf weights for each ranked entity
    ngram_contributions = (
        ngram_weight
        / (rrf_k + np.arange(1, candidates_per_method + 1))
    )

    sapbert_contributions = (
        sapbert_weight
        / (rrf_k + np.arange(1, candidates_per_method + 1))
    )

    for row_idx in range(n_rows):
        fused = {}

        for rank_idx, candidate_idx in enumerate(ngram_indices[row_idx]):
            if candidate_idx == -1:
                continue
            fused[candidate_idx] = (
                fused.get(candidate_idx, 0.0)
                + ngram_contributions[rank_idx]
            )

        for rank_idx, candidate_idx in enumerate(sapbert_indices[row_idx]):
            if candidate_idx == -1:
                continue
            fused[candidate_idx] = (
                fused.get(candidate_idx, 0.0)
                + sapbert_contributions[rank_idx]
            )

        ranked = sorted(
            fused.items(),
            key=lambda item: item[1],
            reverse=True,
        )[:top_k]

        output_indices[row_idx] = [candidate_idx for candidate_idx, _ in ranked]
        output_scores[row_idx] = [score for _, score in ranked]

    return output_indices, output_scores

In [54]:
print(ngram_indices.shape)

(65929, 10)


In [55]:
print(sapbert_indices.shape)

(65929, 10)


In [56]:
rrf_indices, rrf_scores = rrf_top_k(
    ngram_indices=ngram_indices,
    sapbert_indices=sapbert_indices,
    top_k=5,
    rrf_k=3,
    ngram_weight=0.636,
    sapbert_weight=1.0,
)

In [57]:
rrf_indices.shape

(65929, 5)

In [63]:
candidate_lookup = { entity_text:rrf_indices[i].tolist() for i,entity_text in enumerate(entity_texts) }
scores_lookup = { entity_text:rrf_scores[i].tolist() for i,entity_text in enumerate(entity_texts) }

In [64]:
!pip install spacy

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  Using cached spacy-3.8.15-cp310-cp310-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (28 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp310-cp310-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (9.7 kB)
  Using cached preshed-3.0.13-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached thinc-8.3.13-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (14 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.3-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4

Stage 2 reranker

In [65]:
import spacy
from types import SimpleNamespace

nlp = spacy.blank("en")
punct_chars = ['!', '.', '?', '։', '؟', '۔', '܀', '܁', '܂', '߹', '।', '॥', '၊', '။', '።',
                 '፧', '፨', '᙮', '᜵', '᜶', '᠃', '᠉', '᥄', '᥅', '᪨', '᪩', '᪪', '᪫',
                 '᭚', '᭛', '᭞', '᭟', '᰻', '᰼', '᱾', '᱿', '‼', '‽', '⁇', '⁈', '⁉',
                 '⸮', '⸼', '꓿', '꘎', '꘏', '꛳', '꛷', '꡶', '꡷', '꣎', '꣏', '꤯', '꧈',
                 '꧉', '꩝', '꩞', '꩟', '꫰', '꫱', '꯫', '﹒', '﹖', '﹗', '！', '．', '？',
                 '𐩖', '𐩗', '𑁇', '𑁈', '𑂾', '𑂿', '𑃀', '𑃁', '𑅁', '𑅂', '𑅃', '𑇅',
                 '𑇆', '𑇍', '𑇞', '𑇟', '𑈸', '𑈹', '𑈻', '𑈼', '𑊩', '𑑋', '𑑌', '𑗂',
                 '𑗃', '𑗉', '𑗊', '𑗋', '𑗌', '𑗍', '𑗎', '𑗏', '𑗐', '𑗑', '𑗒', '𑗓',
                 '𑗔', '𑗕', '𑗖', '𑗗', '𑙁', '𑙂', '𑜼', '𑜽', '𑜾', '𑩂', '𑩃', '𑪛',
                 '𑪜', '𑱁', '𑱂', '𖩮', '𖩯', '𖫵', '𖬷', '𖬸', '𖭄', '𛲟', '𝪈', '｡', '。', '\n']
nlp.add_pipe("sentencizer", config={"punct_chars": punct_chars})
nlp.max_length = 5000000 # random high enough number to not cause issues
TOKEN_LIMIT = 512

def split_passage_into_sentences(passage):
  doc = nlp(passage.text)

  sentence_passages = []

  for sent in doc.sents:
    sent_start = passage.offset + sent.start_char
    sent_end = passage.offset + sent.end_char

    sent_annos = [anno for anno in passage.annotations
                  if anno.locations[0].offset >= sent_start
                  and anno.locations[0].offset + anno.locations[0].length <= sent_end]


    sentence_passages.append(SimpleNamespace(
        text=sent.text,
        offset=sent_start,
        annotations=sent_annos
    ))

  return sentence_passages

In [5]:
from types import SimpleNamespace
import random
from transformers import AutoTokenizer
import numpy as np

# nlp = spacy.load("en_core_web_sm")
TOKEN_LIMIT = 512

def prep_tokenizer(
      model_name="microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
      ):

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    tokenizer.add_tokens(['[ENTITY]', '[E]', '[/E]', '[TYPES]', '[SEP]', '[SCORE]', ],)

    return tokenizer

def split_passage_into_sentences(passage, nlp):
    doc = nlp(passage.text)

    sentence_passages = []

    for sent in doc.sents:
        sent_start = passage.offset + sent.start_char
        sent_end = passage.offset + sent.end_char

        sent_annos = [anno for anno in passage.annotations
                    if anno.locations[0].offset >= sent_start
                    and anno.locations[0].offset + anno.locations[0].length <= sent_end]


        sentence_passages.append(SimpleNamespace(
            text=sent.text,
            offset=sent_start,
            annotations=sent_annos
        ))

    return sentence_passages

def get_token_count(text, tokenizer):
    return len(
        tokenizer(
            text,
            truncation=False
        )["input_ids"]
    )

def get_anno_label(anno, identifier):
    gold_id = anno.infons.get("concept_id")

    if gold_id is None:
        label = None
    
    else:
        candidate_is_correct = (identifier == gold_id)
        label = ("CORRECT" if candidate_is_correct else "INCORRECT")

    return label

def create_candidates(text, candidate_idxs, candidate_scores, ontology, anno):
    candidates = []
    for idx, score in zip(candidate_idxs, candidate_scores):
            if idx == -1:
                continue
            
            identifier = ontology[idx]['id']
            entity_name = ontology[idx]['name']
            entity_types = "; ".join(ontology[idx]["types"])
            entity_score = min(100, int(score * 100))
            start = len(text)
            end = start + len('[ENTITY]')
            text += f"[ENTITY]{entity_name}[TYPES]{entity_types}[SCORE]{entity_score}"
            label = get_anno_label(anno, identifier)

            candidates.append( {'id':identifier, 'name':entity_name, 'start':start, 'end':end, 'label':label } )
    return text, candidates

def build_annotated_sentence(passage, prev_passage, annos):
    annos = sorted(
        annos,
        key=lambda anno: anno.locations[0].offset
    )

    text = prev_passage + "[SEP]"
    previous_end = 0

    for anno in annos:
        anno_start = anno.locations[0].offset - passage.offset
        anno_end = anno_start + anno.locations[0].length
        text += passage.text[previous_end:anno_start]
        text += f"[E]{passage.text[anno_start:anno_end]}[/E]"

        previous_end = anno_end

    text += passage.text[previous_end:]

    return text

def build_chunk(passage, prev_passage, annos, candidates_by_anno, scores_by_anno, ontology):
    text = build_annotated_sentence(passage, prev_passage, annos)
    candidates = []

    for anno, candidate_idxs, candidate_scores in zip(annos, candidates_by_anno, scores_by_anno):
        text, candidate = create_candidates(text, candidate_idxs, candidate_scores, ontology, anno)
        candidates.extend(candidate)

    return {'text':text, 'candidates':candidates, 'sentence':passage, 'annotations':annos}

def make_candidates_gold(anno, anno_candidate_idxs, anno_candidate_scores, ontology_id_to_idx):
    
    gold_idx = ontology_id_to_idx.get(
            anno.infons["concept_id"]
        )

    if (
        gold_idx is not None
        and gold_idx not in anno_candidate_idxs
    ):
        anno_candidate_idxs[-1] = gold_idx


    paired = list(zip(anno_candidate_idxs, anno_candidate_scores))
    random.shuffle(paired)

    anno_candidate_idxs, anno_candidate_scores = map(
        list,
        zip(*paired)
    )

    return anno_candidate_idxs, anno_candidate_scores


def fast_anno_with_candidates(passage, prev_passage, annos, candidate_lookup, scores_lookup, ontology_id_to_idx, ontology, force_gold):
    annos = sorted(
        annos,
        key=lambda anno: anno.locations[0].offset
    )

    candidate_idxs = [
        candidate_lookup[anno.text]
        for anno in annos
    ]

    candidate_scores = [
        np.asarray(scores_lookup[anno.text], dtype=np.float32)
        for anno in annos
    ]

    candidate_scores = [
        scores / scores.max() if scores.max() > 0 else scores
        for scores in candidate_scores
    ]

    if force_gold:
        for i, anno in enumerate(annos):
            candidate_idxs[i], candidate_scores[i] = make_candidates_gold(anno, 
                                                                        candidate_idxs[i], 
                                                                        candidate_scores[i], 
                                                                        ontology_id_to_idx)

    text_with_entities = build_annotated_sentence(passage, prev_passage, annos)

    candidates = []
    for anno, indices, score in zip(annos, candidate_idxs, candidate_scores):
        text_with_entities, candidate = create_candidates(text_with_entities, indices, score, ontology, anno)
        candidates.extend(candidate)
        
    return {'text':text_with_entities, 'candidates':candidates, 'sentence':passage, 'annotations':annos}

def slow_anno_with_candidates(passage, prev_passage, annos, candidate_lookup, scores_lookup, ontology_id_to_idx, force_gold, tokenizer, ontology):
    # sorted on where they occur in passage
    annos = sorted(
        annos,
        key=lambda anno: anno.locations[0].offset
    )

    all_chunks = []

    current_annos = []
    current_candidate_idxs = []
    current_candidate_scores = []


    for anno in annos:
        anno_candidate_idxs = candidate_lookup[anno.text]

        anno_candidate_scores = np.asarray(scores_lookup[anno.text], dtype=np.float32)

        anno_candidate_scores = (anno_candidate_scores / anno_candidate_scores.max()
                                if anno_candidate_scores.max() > 0 
                                else anno_candidate_scores)

        if force_gold:
            anno_candidate_idxs, anno_candidate_scores = make_candidates_gold(anno, 
                                                                            anno_candidate_idxs, 
                                                                            anno_candidate_scores, 
                                                                            ontology_id_to_idx)

        proposed_annos = current_annos + [anno]
        proposed_candidate_idxs = (current_candidate_idxs + [anno_candidate_idxs])
        proposed_candidate_scores = (current_candidate_scores + [anno_candidate_scores])

        proposed_chunk = build_chunk(passage, prev_passage, proposed_annos, proposed_candidate_idxs, proposed_candidate_scores, ontology)

        if get_token_count(proposed_chunk["text"], tokenizer) <= TOKEN_LIMIT:
            current_annos = proposed_annos
            current_candidate_idxs = proposed_candidate_idxs
            current_candidate_scores = proposed_candidate_scores
            continue

        if current_annos:
            all_chunks.append(build_chunk(passage, prev_passage, current_annos, current_candidate_idxs, current_candidate_scores, ontology))

        current_annos = [anno]
        current_candidate_idxs = [anno_candidate_idxs]
        current_candidate_scores = [anno_candidate_scores]
            
        single_anno_chunk = build_chunk(passage, prev_passage, current_annos, current_candidate_idxs, current_candidate_scores, ontology)

        if get_token_count(single_anno_chunk["text"], tokenizer) > TOKEN_LIMIT:
            for candidate, score in zip(anno_candidate_idxs, anno_candidate_scores):
                single_candidate_chunk = build_chunk(passage, prev_passage, [anno], [[candidate]], [[score]], ontology)
                if get_token_count(single_candidate_chunk["text"], tokenizer) > TOKEN_LIMIT:
                    single_candidate_chunk["text"] = truncate_entity_to_token_limit(single_candidate_chunk["text"], tokenizer)
                all_chunks.append(single_candidate_chunk)

            current_annos = []
            current_candidate_idxs = []
            current_candidate_scores = []
            continue
            
            
        
    if current_annos:
        all_chunks.append(
            build_chunk(
                passage,
                prev_passage,
                current_annos,
                current_candidate_idxs,
                current_candidate_scores,
                ontology
            )
        )

    return all_chunks

def truncate_entity_to_token_limit(text, tokenizer):
    # print(text)
    before_entity, entity_and_after = text.split("[ENTITY]", 1)
    entity_text, after_entity = entity_and_after.rsplit("[TYPES]", 1)

    entity_words = entity_text.split()

    truncated_words = []

    for word in entity_words:
        proposed_words = truncated_words + [word]

        proposed_text = (before_entity
                         + "[ENTITY]"
                         + " ".join(proposed_words)
                         + "[TYPES]"
                         + after_entity
                        )
        if get_token_count(proposed_text, tokenizer) <= TOKEN_LIMIT:
            truncated_words = proposed_words
        else:
            break
    if not truncated_words:
        print(before_entity)
        print(after_entity)
        raise ValueError("token limit exceeded without entity")

    truncated_text = (before_entity
                     + "[ENTITY]"
                     + " ".join(truncated_words)
                     + "[TYPES]"
                     + after_entity
                    )
    
    return truncated_text

    
def make_anno_with_candidates(passage,
    prev_passage,
    annos,
    candidate_lookup,
    scores_lookup,
    ontology,
    ontology_index,
    tokenizer,
    force_gold=False):
    
    candidate_passage = fast_anno_with_candidates(passage, 
                                                  prev_passage, 
                                                  annos, 
                                                  candidate_lookup, 
                                                  scores_lookup, 
                                                  ontology_index, 
                                                  ontology, 
                                                  force_gold)

    if get_token_count(candidate_passage["text"], tokenizer) <= TOKEN_LIMIT:
        return [candidate_passage]
    else:
        return slow_anno_with_candidates(passage, 
                                         prev_passage,
                                         annos, 
                                         candidate_lookup, 
                                         scores_lookup, 
                                         ontology_index, 
                                         force_gold,
                                         tokenizer, 
                                         ontology)

In [76]:
from transformers import AutoTokenizer

model_name = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.add_tokens(['[ENTITY]', '[E]', '[/E]', '[TYPES]', '[SEP]', '[SCORE]', ],)

6

In [77]:
ontology_id_to_idx = {
    concept["id"]: idx
    for idx, concept in enumerate(ontology)
}

In [78]:
train_sentences = [sentence for doc in train_docs for passage in doc.passages for sentence in split_passage_into_sentences(passage, nlp) if len(sentence.annotations) > 0]
val_sentences = [sentence for doc in val_docs for passage in doc.passages for sentence in split_passage_into_sentences(passage, nlp) if len(sentence.annotations) > 0]
test_sentences = [sentence for doc in test_docs for passage in doc.passages for sentence in split_passage_into_sentences(passage, nlp) if len(sentence.annotations) > 0]

In [79]:
import random

train_dataset = [
    chunk
    for i, sentence in enumerate(train_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        train_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        candidate_lookup,
        scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
        force_gold=True
    )
]

val_dataset = [
    chunk
    for i, sentence in enumerate(val_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        val_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        candidate_lookup,
        scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

In [80]:
test_dataset = [
    chunk
    for i, sentence in enumerate(test_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        test_sentences[i-1].text if i>0 else "",
        sentence.annotations,
        candidate_lookup,
        scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

In [ ]:
val_dataset[0]

Tokenize bert

In [101]:
entity_token_id = tokenizer.vocab['[ENTITY]']
entity_token_id

30522

In [102]:
label2id = {'INCORRECT':0, 'CORRECT':1}
id2label = {0:'INCORRECT', 1:'CORRECT'}

In [108]:
def tokenize_and_label(example):
  tokenized = tokenizer(
    example['text'],
    truncation=True,
    max_length=512
  )

  entity_token_locations = [ i for i,token_id in enumerate(tokenized['input_ids']) if token_id == entity_token_id ]

  token_labels = [-100] * len(tokenized['input_ids'])

  for idx,candidate in zip(entity_token_locations, example['candidates']):
    token_labels[idx] = label2id[candidate['label']]

  tokenized['labels'] = token_labels

  return tokenized

In [104]:
from datasets import DatasetDict, Dataset

dataset = DatasetDict({
    "train": Dataset.from_list(train_dataset),
    "validation": Dataset.from_list(val_dataset),
})

ArrowInvalid: Could not convert namespace(text='DCTN4 as a modifier of chronic Pseudomonas aeruginosa infection in cystic fibrosis\n', offset=0, annotations=[BioCAnnotation[id=,text='DCTN4',infons=[concept_id=UMLS:C4308010],locations=[BioCLocation[offset=0,length=5]],], BioCAnnotation[id=,text='chronic Pseudomonas aeruginosa infection',infons=[concept_id=UMLS:C0854135],locations=[BioCLocation[offset=23,length=40]],], BioCAnnotation[id=,text='cystic fibrosis',infons=[concept_id=UMLS:C0010674],locations=[BioCLocation[offset=67,length=15]],]]) with type types.SimpleNamespace: did not recognize Python value type when inferring an Arrow data type

In [105]:
tokenized = dataset.map(tokenize_and_label, remove_columns=dataset["train"].column_names)

NameError: name 'dataset' is not defined

In [100]:
def has_any_labels(example):
  return any(label != -100 for label in example["labels"])
tokenized = dataset.map(tokenize_and_label, remove_columns=dataset["train"].column_names)
tokenized = tokenized.filter(has_any_labels)

NameError: name 'tokenized' is not defined

In [99]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

CANDIDATES_TO_CONSIDER = 5

CORRECT_LABEL_ID = label2id["CORRECT"]
INCORRECT_LABEL_ID = label2id["INCORRECT"]


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    token_predictions = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    candidate_scores = []
    candidate_labels = []

    for sequence_logits, prediction_sequence, label_sequence in zip(
        logits,
        token_predictions,
        labels,
    ):
        for token_logits, predicted_label, gold_label in zip(
            sequence_logits,
            prediction_sequence,
            label_sequence,
        ):
            if gold_label == -100:
                continue

            true_preds.append(int(predicted_label))
            true_labels.append(int(gold_label))

            ranking_score = (
                token_logits[CORRECT_LABEL_ID]
                - token_logits[INCORRECT_LABEL_ID]
            )

            candidate_scores.append(float(ranking_score))
            candidate_labels.append(int(gold_label))

    if len(candidate_scores) % CANDIDATES_TO_CONSIDER != 0:
        raise ValueError(
            "The number of evaluated candidates is not divisible by "
            f"{CANDIDATES_TO_CONSIDER}. Got {len(candidate_scores)} candidates."
        )

    hits = []
    retrieval_failures = 0
    
    for group_start in range(
        0,
        len(candidate_scores),
        CANDIDATES_TO_CONSIDER,
    ):
        group_end = group_start + CANDIDATES_TO_CONSIDER
    
        group_scores = candidate_scores[group_start:group_end]
        group_labels = candidate_labels[group_start:group_end]
    
        gold_positions = [
            index
            for index, label in enumerate(group_labels)
            if label == CORRECT_LABEL_ID
        ]
    
        if len(gold_positions) == 0:
            retrieval_failures += 1
            continue
    
        if len(gold_positions) > 1:
            raise ValueError(
                f"Expected at most one correct candidate in group "
                f"{group_start // CANDIDATES_TO_CONSIDER}, "
                f"but found {len(gold_positions)}."
            )
    
        predicted_position = int(np.argmax(group_scores))
        hits.append(int(predicted_position == gold_positions[0]))
        
    reranker_hits_at_1 = float(np.mean(hits))

    total_groups = (
        len(candidate_scores)
        // CANDIDATES_TO_CONSIDER
    )
    
    stage_1_gold_recall = (
        (total_groups - retrieval_failures)
        / total_groups
    )
    
    return {
        "hits_at_1": reranker_hits_at_1,
        "stage_1_gold_recall": stage_1_gold_recall,
        "retrieval_failures": retrieval_failures,
        "accuracy": accuracy_score(true_labels, true_preds),
        "precision": precision_score(
            true_labels,
            true_preds,
            average="macro",
            zero_division=0,
        ),
        "recall": recall_score(
            true_labels,
            true_preds,
            average="macro",
            zero_division=0,
        ),
        "f1": f1_score(
            true_labels,
            true_preds,
            average="macro",
            zero_division=0,
        ),
    }

In [155]:
from transformers import (
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

search_args = TrainingArguments(
    output_dir="./hyperparameter_search",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="hits_at_1",
    greater_is_better=True,

    learning_rate=1e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,

    max_grad_norm=1.0,
    bf16=True,
    save_total_limit=1,

    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=None,
    model_init=model_init,
    args=search_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0001,
        )
    ],
)
seed=123
data_seed=123

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    id2label=id2label,
    label2id=label2id,
)

model.resize_token_embeddings(len(tokenizer))
args = TrainingArguments(
    output_dir="./tmpdir",

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="hits_at_1",
    greater_is_better=True,

    learning_rate=2.11e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=4,

    weight_decay=0.0822,
    label_smoothing_factor=0.08,

    # Include the best warmup value from Optuna:
    warmup_ratio=0.0344,

    bf16=True,
    save_total_limit=1,
    report_to="none",

    seed=seed,
    data_seed=seed,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0001,
        )
    ],
)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [152]:
def optuna_hp_space(trial):
    return {
        "learning_rate": trial.suggest_float(
            "learning_rate",
            5e-6,
            5e-5,
            log=True,
        ),
        "weight_decay": trial.suggest_float(
            "weight_decay",
            0.02,
            0.15,
        ),
        "warmup_ratio": trial.suggest_float(
            "warmup_ratio",
            0.0,
            0.08,
        ),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size",
            [16, 24, 32],
        ),
        "label_smoothing_factor": trial.suggest_categorical(
            "label_smoothing_factor",
            [0.0, 0.02, 0.05, 0.08, 0.1],
        )
    }

In [93]:
def compute_objective(metrics):
    return metrics["eval_hits_at_1"]

In [156]:
# trainer.train()

Epoch,Training Loss,Validation Loss,Hits At 1,Stage 1 Gold Recall,Retrieval Failures,Accuracy,Precision,Recall,F1
1,0.294500,0.290279,0.859983,0.828300,7016,0.933474,0.873835,0.892019,0.882549
2,0.262000,0.297758,0.863795,0.828300,7016,0.931619,0.867978,0.895383,0.880807
3,0.242300,0.305952,0.860958,0.828300,7016,0.929230,0.862038,0.896639,0.877920
4,0.229800,0.309303,0.864593,0.828300,7016,0.931261,0.866629,0.896779,0.880642


TrainOutput(global_step=4836, training_loss=0.26386471125978983, metrics={'train_runtime': 1272.2575, 'train_samples_per_second': 91.155, 'train_steps_per_second': 3.801, 'total_flos': 2.959176583821096e+16, 'train_loss': 0.26386471125978983, 'epoch': 4.0})

In [157]:
# model.save_pretrained("./saved-model-multi-10")

# # We'll actually set the max length as the BioMedBERT tokenizer doesn't have it set by default
# tokenizer.model_max_length = 512
# tokenizer.save_pretrained("./saved-model-multi-10")

('./saved-model-multi-10/tokenizer_config.json',
 './saved-model-multi-10/special_tokens_map.json',
 './saved-model-multi-10/vocab.txt',
 './saved-model-multi-10/added_tokens.json',
 './saved-model-multi-10/tokenizer.json')

In [88]:
# from transformers import pipeline, AutoModelForTokenClassification, AutoConfig, AutoTokenizer

from transformers import pipeline, AutoModelForTokenClassification

model_path = DATA_DIR / "Models/saved-model-multi-10"  # or HF Hub ID

linker_pipeline = pipeline(
    task="token-classification",
    model=str(model_path),
    tokenizer=str(model_path)
)

Device set to use cuda:0


In [124]:
def predict_candidates(
    dataset,
    linker_pipeline,
    candidate_lookup,
    ontology_id_to_idx,
    batch_size=32,
):
    verifier_candidate_lookup = {}
    verifier_scores_lookup = {}

    texts = [chunk["text"] for chunk in dataset]

    all_predictions = linker_pipeline(texts, batch_size=batch_size,)

    for chunk, predictions in zip(dataset, all_predictions):
        scored_candidates = []

        for candidate in chunk["candidates"]:
            matching_predictions = [prediction for prediction in predictions
                if (
                    prediction["start"] == candidate["start"]
                    and prediction["end"] == candidate["end"]
                )
            ]

            if len(matching_predictions) != 1:
                raise ValueError(
                    f"Expected exactly one prediction for candidate {candidate['name']} at "
                    f"{candidate['start']}:{candidate['end']}, found {len(matching_predictions)}")

            prediction = matching_predictions[0]

            if prediction["entity"] == "CORRECT":
                correct_score = float(prediction["score"])
            else:
                correct_score = 1.0 - float(prediction["score"])

            scored_candidates.append({
                "candidate": candidate,
                "prediction": prediction["entity"],
                "correct_score": correct_score,
            })

        candidate_pos = 0

        for anno in chunk["annotations"]:
            num_candidates = len(candidate_lookup[anno.text])

            anno_candidates = scored_candidates[
                candidate_pos:candidate_pos + num_candidates
            ]

            candidate_pos += num_candidates

            best = max(
                anno_candidates,
                key=lambda result: result["correct_score"],
            )

            candidate = best["candidate"]

            verifier_candidate_lookup[anno.text] = [
                ontology_id_to_idx[candidate["id"]]
            ]

            verifier_scores_lookup[anno.text] = [
                best["correct_score"]
            ]

    return verifier_candidate_lookup, verifier_scores_lookup

In [153]:
train_verifier_candidate_lookup, train_verifier_scores_lookup = predict_candidates(
    train_dataset,
    linker_pipeline,
    candidate_lookup,
    ontology_id_to_idx,
)

val_verifier_candidate_lookup, val_verifier_scores_lookup = predict_candidates(
    val_dataset,
    linker_pipeline,
    candidate_lookup,
    ontology_id_to_idx,
)

test_verifier_candidate_lookup, test_verifier_scores_lookup = predict_candidates(
    test_dataset,
    linker_pipeline,
    candidate_lookup,
    ontology_id_to_idx,
)

In [164]:
train_verifier_dataset = [
    chunk
    for i, sentence in enumerate(train_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        "",
        sentence.annotations,
        train_verifier_candidate_lookup,
        train_verifier_scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

val_verifier_dataset = [
    chunk
    for i, sentence in enumerate(val_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        "",
        sentence.annotations,
        val_verifier_candidate_lookup,
        val_verifier_scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

test_verifier_dataset = [
    chunk
    for i, sentence in enumerate(test_sentences)
    for chunk in make_anno_with_candidates(
        sentence,
        "",
        sentence.annotations,
        test_verifier_candidate_lookup,
        test_verifier_scores_lookup,
        ontology,
        ontology_id_to_idx,
        tokenizer,
    )
]

In [165]:
from collections import Counter

labels = [
    candidate["label"]
    for chunk in val_verifier_dataset
    for candidate in chunk["candidates"]
]

counts = Counter(labels)

print(counts)
print("Total:", len(labels))
print("Correct proportion:", counts["CORRECT"] / len(labels))

Counter({'CORRECT': 23723, 'INCORRECT': 17144})
Total: 40867
Correct proportion: 0.5804928181662466


In [166]:
train_hf = [
    {
        "text": item["text"],
        "candidates": item["candidates"],
    }
    for item in train_verifier_dataset
]

val_hf = [
    {
        "text": item["text"],
        "candidates": item["candidates"],
    }
    for item in val_verifier_dataset
]

In [167]:
dataset = DatasetDict({
    "train": Dataset.from_list(train_hf),
    "validation": Dataset.from_list(val_hf),
})

In [168]:
def has_any_labels(example):
  return any(label != -100 for label in example["labels"])
tokenized = dataset.map(tokenize_and_label, remove_columns=dataset["train"].column_names)
tokenized = tokenized.filter(has_any_labels)

Map:   0%|          | 0/27589 [00:00<?, ? examples/s]

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Filter:   0%|          | 0/27589 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9002 [00:00<?, ? examples/s]

In [169]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

CORRECT_LABEL_ID = label2id["CORRECT"]
INCORRECT_LABEL_ID = label2id["INCORRECT"]


def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    true_preds = []
    true_labels = []

    for prediction_sequence, label_sequence in zip(
        predictions,
        labels,
    ):
        for predicted_label, gold_label in zip(
            prediction_sequence,
            label_sequence,
        ):
            if gold_label == -100:
                continue

            true_preds.append(int(predicted_label))
            true_labels.append(int(gold_label))

    true_preds = np.asarray(true_preds)
    true_labels = np.asarray(true_labels)

    # Treat CORRECT as the positive class:
    # CORRECT = accept the link
    # INCORRECT = reject the link
    precision_correct = precision_score(
        true_labels,
        true_preds,
        pos_label=CORRECT_LABEL_ID,
        zero_division=0,
    )

    recall_correct = recall_score(
        true_labels,
        true_preds,
        pos_label=CORRECT_LABEL_ID,
        zero_division=0,
    )

    f1_correct = f1_score(
        true_labels,
        true_preds,
        pos_label=CORRECT_LABEL_ID,
        zero_division=0,
    )

    # Also measure how well the verifier catches bad links
    precision_incorrect = precision_score(
        true_labels,
        true_preds,
        pos_label=INCORRECT_LABEL_ID,
        zero_division=0,
    )

    recall_incorrect = recall_score(
        true_labels,
        true_preds,
        pos_label=INCORRECT_LABEL_ID,
        zero_division=0,
    )

    f1_incorrect = f1_score(
        true_labels,
        true_preds,
        pos_label=INCORRECT_LABEL_ID,
        zero_division=0,
    )

    # Explicit confusion matrix in [INCORRECT, CORRECT] order
    tn, fp, fn, tp = confusion_matrix(
        true_labels,
        true_preds,
        labels=[INCORRECT_LABEL_ID, CORRECT_LABEL_ID],
    ).ravel()

    return {
        "accuracy": accuracy_score(
            true_labels,
            true_preds,
        ),

        # How reliable are accepted links?
        "correct_precision": precision_correct,

        # How many genuinely correct links survive verification?
        "correct_recall": recall_correct,

        "correct_f1": f1_correct,

        # How reliable are rejected links?
        "incorrect_precision": precision_incorrect,

        # Very important:
        # proportion of genuinely wrong stage-1 links we successfully reject
        "incorrect_recall": recall_incorrect,

        "incorrect_f1": f1_incorrect,

        # Raw counts - useful when analysing what the verifier is doing
        "true_accept": int(tp),
        "false_accept": int(fp),
        "false_reject": int(fn),
        "true_reject": int(tn),
    }

In [170]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    id2label=id2label
)

model.resize_token_embeddings(len(tokenizer))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Embedding(30527, 768, padding_idx=0)

In [171]:
from transformers import (
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

data_collator = DataCollatorForTokenClassification(tokenizer)

search_args = TrainingArguments(
    output_dir="./hyperparameter_search",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="incorrect_f1",
    greater_is_better=True,

    learning_rate=1e-5,
    per_device_train_batch_size=24,
    per_device_eval_batch_size=24,
    num_train_epochs=10,
    weight_decay=0.01,
    warmup_ratio=0.1,

    max_grad_norm=1.0,
    bf16=True,
    save_total_limit=1,

    report_to="none",
    seed=42,
    data_seed=42,
)

trainer = Trainer(
    model=model,
    args=search_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.0001,
        )
    ],
)

In [172]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Correct Precision,Correct Recall,Correct F1,Incorrect Precision,Incorrect Recall,Incorrect F1,True Accept,False Accept,False Reject,True Reject
1,0.488800,0.587630,0.724545,0.688616,0.959238,0.801705,0.876359,0.399790,0.549089,22756,10290,967,6854
2,0.380800,0.602205,0.736903,0.694252,0.977069,0.811732,0.927273,0.404573,0.563353,23179,10208,544,6936
3,0.319200,0.646741,0.750801,0.706810,0.975256,0.819612,0.927834,0.440212,0.597120,23136,9597,587,7547
4,0.263400,0.600808,0.769129,0.726967,0.964549,0.829073,0.910446,0.498717,0.644432,22882,8594,841,8550
5,0.230500,0.624684,0.775907,0.735109,0.959828,0.832572,0.903660,0.521407,0.661266,22770,8205,953,8939
6,0.203300,0.719504,0.769888,0.726373,0.968385,0.830099,0.918831,0.495217,0.643572,22973,8654,750,8490
7,0.180500,0.858772,0.757726,0.713026,0.975087,0.823716,0.929852,0.456953,0.612773,23132,9310,591,7834


TrainOutput(global_step=8050, training_loss=0.299618042774082, metrics={'train_runtime': 1033.7307, 'train_samples_per_second': 266.888, 'train_steps_per_second': 11.125, 'total_flos': 3.261457124436864e+16, 'train_loss': 0.299618042774082, 'epoch': 7.0})

In [173]:
model.save_pretrained("Models/verifier-2")

# We'll actually set the max length as the BioMedBERT tokenizer doesn't have it set by default
tokenizer.model_max_length = 512
tokenizer.save_pretrained("Models/verifier-2")

('Models/verifier-2/tokenizer_config.json',
 'Models/verifier-2/special_tokens_map.json',
 'Models/verifier-2/vocab.txt',
 'Models/verifier-2/added_tokens.json',
 'Models/verifier-2/tokenizer.json')

In [128]:
def predict_labels(dataset_instance):
  return linker_pipeline(dataset_instance['text'])

In [129]:
def predict_by_coords(predicted_labels):
  predictions_by_coordinates = { (pl['start'],pl['end']):pl for pl in predicted_labels }
  return predictions_by_coordinates

In [130]:
import math

def probability_to_logit(probability, epsilon=1e-7):
    probability = min(
        max(probability, epsilon),
        1.0 - epsilon,
    )

    return math.log(probability / (1.0 - probability))


def softmax(values):
    values = np.asarray(values, dtype=np.float64)

    shifted_values = values - np.max(values)
    exponentials = np.exp(shifted_values)

    return exponentials / exponentials.sum()

In [131]:
CANDIDATES_TO_CONSIDER=5

In [132]:
def evaluate_reranker_with_confidence(
    scores,
    k=1,
    group_size=CANDIDATES_TO_CONSIDER,
    temperature=1.5,
):
    validate_evaluation_hits(scores, k, group_size)

    hits = []
    predictions = []
    retrieval_failures = 0

    all_group_logits = []
    all_gold_positions = []

    for group_start in range(0, len(scores), group_size):
        group = scores[
            group_start:group_start + group_size
        ]

        if len(group) != group_size:
            raise ValueError(
                f"Incomplete candidate group at index {group_start}: "
                f"expected {group_size}, got {len(group)}."
            )

        gold_positions = [
            i
            for i, candidate in enumerate(group)
            if candidate["label"] == "CORRECT"
        ]

        if not gold_positions:
            retrieval_failures += 1
            continue

        if len(gold_positions) > 1:
            raise ValueError(
                f"Multiple gold candidates found in group "
                f"starting at index {group_start}."
            )

        candidate_logits = [
            float(candidate["ranking_logit"])
            for candidate in group
        ]

        gold_position = gold_positions[0]

        all_group_logits.append(candidate_logits)
        all_gold_positions.append(gold_position)

        candidate_confidences = softmax(
            np.asarray(candidate_logits) / temperature
        )

        ranked_indices = np.argsort(
            candidate_confidences
        )[::-1]

        winner_index = int(ranked_indices[0])
        runner_up_index = int(ranked_indices[1])

        winner = group[winner_index]
        runner_up = group[runner_up_index]

        winner_confidence = float(
            candidate_confidences[winner_index]
        )

        runner_up_confidence = float(
            candidate_confidences[runner_up_index]
        )

        confidence_margin = (
            winner_confidence - runner_up_confidence
        )

        correct_in_k = any(
            group[int(index)]["label"] == "CORRECT"
            for index in ranked_indices[:k]
        )

        hits.append(int(correct_in_k))

        predictions.append({
            "predicted_id": winner["id"],
            "predicted_name": winner["name"],
            "is_correct": (
                winner["label"] == "CORRECT"
            ),
            "winner_confidence": winner_confidence,
            "runner_up_id": runner_up["id"],
            "runner_up_name": runner_up["name"],
            "runner_up_confidence": (
                runner_up_confidence
            ),
            "confidence_margin": confidence_margin,
            "candidate_confidences": [
                {
                    "id": candidate["id"],
                    "name": candidate["name"],
                    "gold_label": candidate["label"],
                    "raw_correct_score": candidate[
                        "correct_score"
                    ],
                    "ranking_logit": candidate[
                        "ranking_logit"
                    ],
                    "group_confidence": float(
                        confidence
                    ),
                }
                for candidate, confidence in zip(
                    group,
                    candidate_confidences,
                )
            ],
        })

    if not predictions:
        raise ValueError(
            "No candidate groups contained the gold concept."
        )

    metrics = {
        f"reranker_hits_at_{k}": (
            sum(hits) / len(hits)
        ),
        "mean_winner_confidence": float(np.mean([
            prediction["winner_confidence"]
            for prediction in predictions
        ])),
        "mean_confidence_margin": float(np.mean([
            prediction["confidence_margin"]
            for prediction in predictions
        ])),
        "evaluated_groups": len(predictions),
        "retrieval_failures_excluded": (
            retrieval_failures
        ),
        "stage_1_gold_recall": (
            len(predictions)
            / (
                len(predictions)
                + retrieval_failures
            )
        ),
    }

    group_logits = torch.tensor(
        all_group_logits,
        dtype=torch.float32,
    )

    gold_positions = torch.tensor(
        all_gold_positions,
        dtype=torch.long,
    )

    return (
        metrics,
        predictions,
        group_logits,
        gold_positions,
    )

In [133]:
def get_scores(dataset, predictions_by_coordinates):

  scores = []

  for c in dataset['candidates']:
    pl = predictions_by_coordinates[(c['start'],c['end'])]

    label_score = float(pl['score'])
    correct_score = label_score if pl['entity'] == 'CORRECT' else (1.0 - label_score)

    ranking_logit = probability_to_logit(
            correct_score
        )

    scores.append({'correct_score':correct_score, 'ranking_logit':ranking_logit, 'id':c['id'], 'name':c['name'], 'label':c['label']})
    
  return scores

In [134]:
def get_scores_for_dataset(dataset, batch_size = 16):
    texts = [instance["text"] for instance in dataset]

    all_predictions = linker_pipeline(texts, batch_size=batch_size)
    
    scores = []

    for instance, predicted_labels in tqdm(
        zip(dataset, all_predictions),
        total=len(dataset),
        desc="Extracting candidate scores"
    ):
        predictions_by_coordinates = { (pl['start'],pl['end']):pl for pl in predicted_labels }
        scores.extend(get_scores(instance, predictions_by_coordinates))

    return scores

In [135]:
def evaluate_reranker_with_confidence( 
    scores, 
    k=1, 
    group_size=CANDIDATES_TO_CONSIDER, 
): 
    validate_evaluation_hits(scores, k, group_size) 
    hits = [] 
    predictions = [] 
    retrieval_failures = 0 
    
    all_group_logits = [] 
    all_gold_positions = [] 
    
    for group_start in range(0, len(scores), group_size): 
        group = scores[group_start:group_start + group_size] 
        
        if len(group) != group_size: 
            raise ValueError( 
                f"Incomplete candidate group at index {group_start}: " 
                f"expected {group_size}, got {len(group)}." 
            ) 
        # Only evaluate the reranker when Stage 1 retrieved the gold CUI. 
        gold_is_present = any( 
            candidate["label"] == "CORRECT" 
            for candidate in group ) 
        
        if not gold_is_present: 
            retrieval_failures += 1 
            continue 
        
        candidate_logits = [ 
            probability_to_logit(candidate["correct_score"]) 
            for candidate in group ] 
        
        gold_position = next( 
            i for i, candidate in enumerate(group) 
            if candidate["label"] == "CORRECT" 
        ) 
        
        all_group_logits.append(candidate_logits) 
        all_gold_positions.append(gold_position) 
        
        candidate_confidences = softmax(np.array(candidate_logits)/ 2.49) 
        
        ranked_indices = np.argsort(candidate_confidences)[::-1] 
        
        winner_index = int(ranked_indices[0]) 
        
        runner_up_index = int(ranked_indices[1]) 
        
        winner = group[winner_index] 
        
        runner_up = group[runner_up_index] 
        
        winner_confidence = float( candidate_confidences[winner_index] ) 
        
        runner_up_confidence = float( candidate_confidences[runner_up_index] ) 
        
        confidence_margin = ( winner_confidence - runner_up_confidence ) 
        
        correct_in_k = any( 
            group[int(index)]["label"] == "CORRECT" 
            for index in ranked_indices[:k] 
        ) 
        
        hits.append(int(correct_in_k)) 
        predictions.append({ 
            "predicted_id": winner["id"], 
            "predicted_name": winner["name"], 
            "is_correct": winner["label"] == "CORRECT", 
            "winner_confidence": winner_confidence, 
            "runner_up_id": runner_up["id"], 
            "runner_up_name": runner_up["name"], 
            "runner_up_confidence": runner_up_confidence, 
            "confidence_margin": confidence_margin, 
            "candidate_confidences": [ { 
                "id": candidate["id"], 
                "name": candidate["name"], 
                "gold_label": candidate["label"], 
                "raw_correct_score": candidate["correct_score"], 
                "group_confidence": float(confidence), 
            } 
                for candidate, confidence in zip( 
                    group, 
                    candidate_confidences, 
                ) 
             ], 
        }) 
        
    if not predictions: 
        raise ValueError( "No candidate groups contained the gold concept." ) 
    
    metrics = { 
        f"reranker_hits_at_{k}": sum(hits) / len(hits), 
        "mean_winner_confidence": float(np.mean([ 
            prediction["winner_confidence"] 
            for prediction in predictions ])), 
        "mean_confidence_margin": float(np.mean([ 
            prediction["confidence_margin"] 
            for prediction in predictions ])), 
        "evaluated_groups": len(predictions), 
        "retrieval_failures_excluded": retrieval_failures, 
        "stage_1_gold_recall": ( 
            len(predictions) 
            / (len(predictions) + retrieval_failures) ), } 
    
    val_group_logits = torch.tensor( all_group_logits, dtype=torch.float32, ) 
    
    val_gold_positions = torch.tensor( all_gold_positions, dtype=torch.long, ) 
    
    return metrics, predictions, val_group_logits, val_gold_positions

In [136]:
def validate_evaluation_hits(dataset, k, group_size):
  if k < 1:
    raise Exception("the number of k to evaluate at must be at least 1")

  if k>group_size:
    raise Exception("the number of k to evaluate at must be at most number of candidates")

  if dataset[0] == None:
    raise Exception("The dataset is empty")

  return

In [137]:
def hits_at_k_stage1(dataset, k=5, group_size=10):

    validate_evaluation_hits(dataset, k, group_size)

    hits = []

    for data in dataset:
        candidates = data["candidates"]

        for start in range(0, len(candidates), group_size):
          candidate_group = candidates[start:start+k]

          hit = any(
              candidate["label"] == "CORRECT"
              for candidate in candidate_group
          )

          hits.append(hit)

    return sum(hits) / len(hits)

In [138]:
def hits_at_k_overall(dataset, k=1, group_size=CANDIDATES_TO_CONSIDER):
    # scores = get_scores_for_dataset(dataset)

    validate_evaluation_hits(dataset, k, group_size)


    hits = []
    group_counter = 0
    group_scores = []
    for score in dataset:
      group_counter += 1

      group_scores.append(score)

      if group_counter == group_size:
        correct_in_k = False
        group_scores = sorted(group_scores, key=lambda x: x['correct_score'], reverse=True)
        for i in range(k):
          if group_scores[i]['label'] == 'CORRECT':
            correct_in_k = True
            break
        hits.append(1 if correct_in_k else 0)

        group_counter = 0
        group_scores = []

    return sum(hits) / len(hits)

In [139]:
def hits_at_k_stage2(dataset, k=1):
  scores = get_scores_for_dataset(dataset,
    batch_size=16,)
  overall_hits = hits_at_k_overall(scores, k)
  stage1_hits = hits_at_k_stage1(dataset, CANDIDATES_TO_CONSIDER)
  print("Overall: ",  overall_hits)
  print("stage 1: ",  stage1_hits)
  return overall_hits / stage1_hits


In [230]:
# test_scores

In [ ]:
# scores = get_scores(test_scores)
print("Overall train:", hits_at_k_overall(get_scores_for_dataset(train_dataset)))
print("Overall val:", hits_at_k_overall(get_scores_for_dataset(val_dataset)))
# print("Overall test:", hits_at_k_overall(get_scores_for_dataset(test_dataset)))
# hits_at_k_overall(get_scores_for_dataset(test_dataset))
# test_scores

In [151]:
print("Stage 1 train Hits@5:", hits_at_k_stage1(train_dataset, k=5))
print("Stage 1 val Hits@5:", hits_at_k_stage1(val_dataset, k=5))
print("Stage 1 test Hits@5:", hits_at_k_stage1(test_dataset, k=5))

Stage 1 train Hits@5: 0.8768600041985157
Stage 1 val Hits@5: 0.6548187899728001
Stage 1 test Hits@5: 0.6405779874447457


In [ ]:
# print("Stage 2 train Hits@1:", hits_at_k_stage2(train_dataset))
# print("Stage 2 val Hits@1:", hits_at_k_stage2(val_dataset))
print("Stage 2 test Hits@1:", hits_at_k_stage2(test_dataset))

In [150]:
val_scores = get_scores_for_dataset(val_dataset)

KeyboardInterrupt: 

In [ ]:
import torch

val_metrics, val_predictions, val_group_logits, val_gold_positions = evaluate_reranker_with_confidence(
    val_scores,
    k=1,
    group_size=CANDIDATES_TO_CONSIDER,
)

print(val_metrics)

In [146]:
labels = [
    candidate["label"]
    for chunk in val_verifier_dataset
    for candidate in chunk["candidates"]
]

correct = sum(label == "CORRECT" for label in labels)
incorrect = sum(label == "INCORRECT" for label in labels)

print("Total:", len(labels))
print("Correct:", correct)
print("Incorrect:", incorrect)
print("Correct proportion:", correct / len(labels))

Total: 40867
Correct: 23723
Incorrect: 17144
Correct proportion: 0.5804928181662466


In [149]:
labels = [
    candidate["label"]
    for chunk in val_dataset
    for candidate in chunk["candidates"]
]

correct = sum(label == "CORRECT" for label in labels)
incorrect = sum(label == "INCORRECT" for label in labels)

print("Total:", len(labels))
print("Correct:", correct)
print("Incorrect:", incorrect)
print("Correct proportion:", correct / len(labels))

Total: 204335
Correct: 30240
Incorrect: 174095
Correct proportion: 0.14799226759977488


In [191]:
correct_confidences = [
    prediction["winner_confidence"]
    for prediction in val_predictions
    if prediction["is_correct"]
]

incorrect_confidences = [
    prediction["winner_confidence"]
    for prediction in val_predictions
    if not prediction["is_correct"]
]

correct_margins = [
    prediction["confidence_margin"]
    for prediction in val_predictions
    if prediction["is_correct"]
]

incorrect_margins = [
    prediction["confidence_margin"]
    for prediction in val_predictions
    if not prediction["is_correct"]
]

print(
    "Mean confidence when correct:",
    np.mean(correct_confidences),
)

print(
    "Mean margin when correct:",
    np.mean(correct_margins),
)

print(
    "Mean confidence when incorrect:",
    np.mean(incorrect_confidences),
)

print(
    "Mean margin when incorrect:",
    np.mean(incorrect_margins),
)

Mean confidence when correct: 0.8967563058875363
Mean margin when correct: 0.8572209004706951
Mean confidence when incorrect: 0.7082697397627572
Mean margin when incorrect: 0.5656456084754077


In [192]:
def confidence_band_accuracy(
    predictions,
    band_width=0.1,
):
    results = []

    lower_bound = 0.0

    while lower_bound < 1.0:
        upper_bound = lower_bound + band_width

        band_predictions = [
            prediction
            for prediction in predictions
            if (
                lower_bound
                <= prediction["winner_confidence"]
                < upper_bound
            )
        ]

        if band_predictions:
            accuracy = np.mean([
                prediction["is_correct"]
                for prediction in band_predictions
            ])

            results.append({
                "confidence_range": (
                    f"{lower_bound:.1f}-"
                    f"{upper_bound:.1f}"
                ),
                "count": len(band_predictions),
                "accuracy": float(accuracy),
            })

        lower_bound = upper_bound

    return results

In [193]:
confidence_bands = confidence_band_accuracy(
    val_predictions
)

for band in confidence_bands:
    print(band)

{'confidence_range': '0.2-0.3', 'count': 117, 'accuracy': 0.4444444444444444}
{'confidence_range': '0.3-0.4', 'count': 503, 'accuracy': 0.4274353876739563}
{'confidence_range': '0.4-0.5', 'count': 1103, 'accuracy': 0.4814143245693563}
{'confidence_range': '0.5-0.6', 'count': 1210, 'accuracy': 0.5206611570247934}
{'confidence_range': '0.6-0.7', 'count': 1305, 'accuracy': 0.5885057471264368}
{'confidence_range': '0.7-0.8', 'count': 1796, 'accuracy': 0.6854120267260579}
{'confidence_range': '0.8-0.9', 'count': 3812, 'accuracy': 0.776495278069255}
{'confidence_range': '0.9-1.0', 'count': 24000, 'accuracy': 0.9532916666666666}


In [185]:
import torch
import torch.nn as nn


class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()

        # Optimise log-temperature so temperature always remains positive.
        self.log_temperature = nn.Parameter(torch.zeros(1))

    @property
    def temperature(self):
        return self.log_temperature.exp()

    def forward(self, logits):
        return logits / self.temperature

In [186]:
def fit_temperature(
    val_group_logits,
    val_gold_positions,
    max_iter=100,
):
    """
    val_group_logits:
        Tensor of shape [number_of_groups, candidates_per_group]

    val_gold_positions:
        Tensor of shape [number_of_groups]
        containing the correct candidate position for each group.
    """

    val_group_logits = val_group_logits.detach().float()
    val_gold_positions = val_gold_positions.detach().long()

    scaler = TemperatureScaler()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.LBFGS(
        scaler.parameters(),
        lr=0.01,
        max_iter=max_iter,
    )

    def closure():
        optimizer.zero_grad()

        scaled_logits = scaler(val_group_logits)
        loss = criterion(scaled_logits, val_gold_positions)

        loss.backward()
        return loss

    optimizer.step(closure)

    return scaler.temperature.item()

In [187]:
temperature = fit_temperature(
    val_group_logits,
    val_gold_positions,
)
print("Learned temperature:", temperature)

Learned temperature: 1.2875570058822632
